# 05 · C1/C2/C3 — raw formats and the ingestion reality check

§5.2 claims the system ingests real resumes in their original formats. This notebook opens
actual PDFs and actual HTML and records **what the parser will have to survive** — plus the
contamination check that keeps this corpus from leaking into the A1 benchmark.

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from candidate_screener.config import DOCS_DATA, RAW, MANIFEST, PROFILE_METRICS
from candidate_screener.data.profile import load_split
from candidate_screener.data.sources import SOURCES

pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

METRICS = json.loads(PROFILE_METRICS.read_text())["checks"]
print("profile metrics generated:", json.loads(PROFILE_METRICS.read_text())["generated"])

profile metrics generated: 2026-08-19T11:44:13+00:00


In [2]:
import pypdf

root = RAW / "snehaanbhawal-resume-dataset"
csv = pd.read_csv(root / "Resume.csv")
pdfs = sorted(root.rglob("*.pdf"))
print(f"{len(csv)} rows, {len(pdfs)} PDFs across {len({p.parent.name for p in pdfs})} categories")
csv.Category.value_counts().head(10)

2484 rows, 2484 PDFs across 24 categories


Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
Name: count, dtype: int64

## Open real PDFs — page counts, extraction yield, and where it goes wrong

In [3]:
def probe(path):
    r = pypdf.PdfReader(path)
    text = "\n".join(p.extract_text() or "" for p in r.pages)
    return {"file": path.name, "category": path.parent.name, "KB": path.stat().st_size // 1024,
            "pages": len(r.pages), "chars": len(text), "chars/page": len(text) // max(1, len(r.pages)),
            "empty pages": sum(1 for p in r.pages if not (p.extract_text() or "").strip())}

probes = pd.DataFrame([probe(p) for p in pdfs[:40]])
print(probes[["pages", "chars", "chars/page", "empty pages"]].describe().round(0))
probes.head(6)

       pages    chars  chars/page  empty pages
count   40.0     40.0        40.0         40.0
mean     2.0   6326.0      3133.0          0.0
std      1.0   3434.0       993.0          0.0
min      1.0   1641.0      1641.0          0.0
25%      2.0   4988.0      2522.0          0.0
50%      2.0   5817.0      2986.0          0.0
75%      2.0   6661.0      3393.0          0.0
max      5.0  24295.0      6416.0          0.0


,file,category,KB,pages,chars,chars/page,empty pages
0,10554236.pdf,ACCOUNTANT,47,5,24295,4859,0
1,10674770.pdf,ACCOUNTANT,24,2,7567,3783,0
2,11163645.pdf,ACCOUNTANT,21,2,4838,2419,0
3,11759079.pdf,ACCOUNTANT,22,2,5986,2993,0
4,12065211.pdf,ACCOUNTANT,23,2,5622,2811,0
5,12202337.pdf,ACCOUNTANT,22,2,5301,2650,0


In [4]:
sample = pdfs[0]
text = "\n".join(p.extract_text() or "" for p in pypdf.PdfReader(sample).pages)
print(f"=== {sample.parent.name}/{sample.name}\n")
print(text[:1200])

=== ACCOUNTANT/10554236.pdf

ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented
Financial reporting
Critical thinking
Accounting operations professional
Analysis of financial systems
ERP (Enterprise Resource Planning) software.
Excellent facilitator
Accomplishments
Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. This allowed
for the first successful fiscal year-end close for 2012.
In collaboration with DFAS Europe, developed an automated tool that identified duplicate obligations. This tool allowed HQ USAFE to
deobligate over $5M in duplicate obligations.
Experience
Company Name
 
July 2011
 
to 
November 2012
 
Accountant
 
City
 
, 
State
Enterprise Resource Planning Office (ERO)
In this position as an Accountant assigned to the Defense Enterprise Accounting and Management 

### Failure modes the ingestion component must handle

Read the extraction above against the rendered document. What the raw text stream gives you:

1. **No section structure.** Headings arrive as ordinary lines; `Summary`, `Skills`,
   `Experience` are recoverable only by heuristics or a layout model.
2. **Multi-column and table layouts interleave.** Text is emitted in draw order, so a
   two-column resume can interleave a skills sidebar into the experience narrative — the
   sentence boundaries a span extractor assumes are not reliable.
3. **Dates and bullets lose their anchoring**, which is what a `Years of Experience`
   extractor needs most.
4. **Some pages extract empty** (scanned/vector text) — count them and route to OCR or fail
   loudly rather than silently returning a short resume.

In [5]:
row = csv.iloc[0]
extracted = "\n".join(p.extract_text() or "" for p in pypdf.PdfReader(
    next(p for p in pdfs if p.stem == str(row.ID))).pages)
print(f"ID {row.ID} | {row.Category}")
print(f"  chars — publisher Resume_str: {len(row.Resume_str)}   our pypdf extraction: {len(extracted)}")
print(f"  Resume_html: {len(row.Resume_html)} chars")
print("\n--- publisher Resume_str (first 400)\n", row.Resume_str[:400])

ID 16852973 | HR
  chars — publisher Resume_str: 5442   our pypdf extraction: 5179
  Resume_html: 16611 chars

--- publisher Resume_str (first 400)
          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team manageme


The corpus ships three views of the same document — **PDF**, publisher-extracted
`Resume_str`, and `Resume_html`. That is what makes it usable as a *parser test set*:
`Resume_str` is a reference extraction to diff our own pipeline against.

## C2 — the HTML mirror (no Kaggle account needed)

In [6]:
html_df = load_split("livecareer", "train")
h = html_df.Resume_html.iloc[0]
print(f"{len(html_df)} rows; median HTML {int(html_df.Resume_html.str.len().median()):,} chars")
tags = pd.Series(pd.Series(h.split("<")).str.extract(r"^(/?[a-zA-Z0-9]+)")[0].dropna()).value_counts()
print("tag inventory of one record:", tags.head(12).to_dict())
print(h[:500])

2484 rows; median HTML 15,025 chars
tag inventory of one record: {'span': 129, '/span': 129, 'div': 48, '/div': 48, 'li': 30, '/li': 30, 'p': 11, '/p': 11, 'br': 9, 'ul': 8, '/ul': 8, 'u': 7}
<div class="fontsize fontface vmargins hmargins linespacing pagesize" id="document"> <div class="section firstsection" id="SECTION_NAME500375979" style="
      padding-top:0px;
    "> <div class="paragraph PARAGRAPH_NAME firstparagraph" id="PARAGRAPH_500375979_1_326506904" style="
      padding-top:0px;
    "> <div class="name" itemprop="name"> <span class="field fName" id="500375979FNAM1"> </span> <span> </span> <span class="field" id="500375979LNAM1"> HR ADMINISTRATOR/MARKETING ASSOCIATE

HR A


Real markup: nested `div`/`span` layout tables, inline styles, no semantic sectioning. An
HTML-to-text step that strips tags naively concatenates words across cells — the same
interleaving failure as the PDF path, so both need the same whitespace/segmentation repair.

## Contamination — 44 of these resumes are also in the A1 benchmark

In [7]:
excl = pd.read_csv("../data/interim/c1_a1_contamination.csv")
print(f"{len(excl)} resumes flagged at >=0.70 8-gram containment against A1")
print("by threshold:", METRICS["livecareer"]["a1_overlap_by_threshold"])
excl.head(10)

44 resumes flagged at >=0.70 8-gram containment against A1
by threshold: {'0.9': 2, '0.7': 44, '0.5': 65}


,ID,Category,containment
0,10554236,ACCOUNTANT,0.925
1,24670867,FINANCE,0.922
2,34816637,ACCOUNTANT,0.885
3,16237710,ACCOUNTANT,0.870
4,28359817,ACCOUNTANT,0.869
5,25862026,ACCOUNTANT,0.866
6,12802330,ACCOUNTANT,0.866
7,15821633,ACCOUNTANT,0.858
8,64468610,ENGINEERING,0.858
9,28078163,ENGINEERING,0.856


Exact matching finds **zero** — the two scrapes format differently — so the check runs on
8-gram containment. The list is written to `data/interim/c1_a1_contamination.csv` by
`profile --check livecareer` and **must be excluded** from any NER training set or
distractor pool built off this corpus, or those documents leak into the matching evaluation.

## C3 ResumeAtlas — confirm it is pre-normalised, then set it aside

In [8]:
atlas = load_split("resume-atlas", "train")
print(f"{len(atlas)} rows, {atlas.Category.nunique()} categories")
print(atlas.Text.iloc[0][:400])

13389 rows, 43 categories
education omba executive leadership university texas 20162018 bachelor science accounting richland college 20052008 training certifications certified management accountant cma certified financial modeling valuation analyst compliance antimoney laundering 092016 american institute banking certified public account cpa lean six sigma green belt certified trade products financial regulations 082016 am


Lowercased, punctuation-stripped, stopwords removed. No case, no punctuation, no character
offsets — therefore **no NER, no evidence spans, nothing displayable in the recruiter UI**.
Restricted to the §4.7 clustering EDA.